# 1. Implementação do Levenberg-Marquardt

In [14]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

dados = pd.read_excel("Trabalho2dados.xlsx")
x_raw = dados["x"].to_numpy()
y_raw = dados["y"].to_numpy()
z_raw = dados["z"].to_numpy()

# Funções utilizadas anteriormente
def zscore_normaliza(v):
    v_mean, v_std = v.mean(), v.std()
    return (v - v_mean) / v_std, v_mean, v_std

def zscore_desnormaliza(v_norm, v_mean, v_std):
    return v_norm * v_std + v_mean

def construir_F_bar(xn, yn):
    return np.column_stack((xn**3, yn**2, np.ones(len(xn))))

x, x_mean, x_std = zscore_normaliza(x_raw)
y, y_mean, y_std = zscore_normaliza(y_raw)
z, z_mean, z_std = zscore_normaliza(z_raw)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

F_zscore = construir_F_bar(x, y)
n, p = F_zscore.shape

def modelo(theta, F):
    return F @ theta

def erro(theta, F, z):
    return z - modelo(theta, F)

def J_MSE(theta, F, z):
    e = erro(theta, F, z)
    return np.mean(e**2)

def gradiente_J_MSE(theta, F, z):
    e = erro(theta, F, z)
    return - (2 / n) * (F.T @ e)


# Implementação de Levengerg-Marquardt

def levenberg_marquardt(F, z, x0, alpha0=1e-3, delta_alpha=10.0, max_iter=10000, tol=1e-8, alpha_max=1e12):
    w = np.array(x0, dtype=float)
    n, p = F.shape
    alpha_lm = alpha0

    x_hist, hist = [], []
    J_k = J_MSE(w, F, z)
    hist.append(f"Iteração 0 : w = {w} -> MSE = {J_k:.6f}\n")
    x_hist.append(w.copy())

    J_ww = (2 / n) * (F.T @ F)

    for i in range(max_iter):
        e = erro(w, F, z)
        J_w = - (2 / n) * (F.T @ e)

        J_aux = np.inf
        delta_w = np.zeros(p)
        while J_aux >= J_k and alpha_lm <= alpha_max:
            H_lm = J_ww + alpha_lm * np.eye(p)
            delta_w = np.linalg.solve(H_lm, -J_w)
            w_aux = w + delta_w
            J_aux = J_MSE(w_aux, F, z) 
            if J_aux >= J_k:
                alpha_lm *= delta_alpha

        if J_aux < J_k:
            delta_J = J_k - J_aux
            alpha_lm /= delta_alpha
            w, J_k = w_aux, J_aux
        else:
            delta_J = 0.0        # dispara a parada logo abaixo

        hist.append(f"Iteração {i+1} : w = {w} -> MSE = {J_k:.6f} (alpha = {alpha_lm:.1e})\n")
        x_hist.append(w.copy())

        if np.linalg.norm(delta_w) <= tol or delta_J <= tol:
            hist[-1] = hist[-1].replace("\n", " [Convergiu]\n")
            return w, hist, x_hist

    return w, hist, x_hist

x0 = [0.0, 0.0, 0.0]

theta, hist, x_hist = levenberg_marquardt(F_zscore, z, x0)

z_pred_lm = zscore_desnormaliza(modelo(theta, F_zscore), z_mean, z_std)

r2_lm = r2_score(z_raw, z_pred_lm)

iters_lm = len(x_hist) - 1

print(f"Iterações {iters_lm}")

convergiu = "[Convergiu]" in hist[-1]
print(f"{'levenberg Marquardt':<5} -> {len(x_hist) - 1:5d} iterações " f"(convergiu={convergiu}) -> theta* = {theta} -> J(theta*) = {J_MSE(theta, F_zscore, z):.6f}")

Iterações 3
levenberg Marquardt ->     3 iterações (convergiu=True) -> theta* = [ 0.51128731  0.18732036 -0.18732036] -> J(theta*) = 0.014243


# 2. Comparação dos métodos

In [16]:

def J_RMSE(theta, F_bar, z):
    return np.sqrt(J_MSE(theta, F_bar, z))

def gradiente_J_RMSE(theta, F_bar, z):
    rmse = J_RMSE(theta, F_bar, z)
    if rmse == 0:
        return np.zeros_like(theta)
    return gradiente_J_MSE(theta, F_bar, z) / (2 * rmse)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)


def descida_gradiente(J, gradiente_J, alpha, x0, args=(), max_iter=10000, tol=1e-8):
    x = np.array(x0, dtype=float)
    x_hist, hist = [], []
    
    Jx = J(x, *args)
    hist.append(f"Iteração 0 : x = {x} -> J(x) = {Jx:.6f}\n")
    x_hist.append(x.copy())
    
    for i in range(max_iter):
        grad = gradiente_J(x, *args)
        xi = x - (alpha * grad)
        
        Jxi = J(xi, *args)
        Jx = J(x, *args)
        
        if np.linalg.norm(xi - x) <= tol or abs(Jxi - Jx) <= tol:
            hist.append(f"Iteração {i + 1} : x = {xi} -> J(x) = {Jxi:.6f} [Convergiu]\n")
            x_hist.append(xi.copy())
            return xi, hist, x_hist
            
        x = xi
        hist.append(f"Iteração {i + 1} : x = {x} -> J(x) = {Jxi:.6f}\n")
        x_hist.append(x.copy())
        
    return x, hist, x_hist

x0 = [0.0, 0.0, 0.0]
alpha_gd = 0.05 

theta_gd, hist_gd, x_hist_gd = descida_gradiente( J_RMSE, gradiente_J_RMSE, alpha_gd, x0, args=(F_zscore, z))

theta_lm, hist_lm, x_hist_lm = levenberg_marquardt(F_zscore, z, x0, alpha0=1e-3)

z_pred_gd = zscore_desnormaliza(modelo(theta_gd, F_zscore), z_mean, z_std)
z_pred_lm = zscore_desnormaliza(modelo(theta_lm, F_zscore), z_mean, z_std)

r2_gd = r2_score(z_raw, z_pred_gd)
r2_lm = r2_score(z_raw, z_pred_lm)

iters_gd = len(x_hist_gd) - 1
iters_lm = len(x_hist_lm) - 1

print("=" * 60)
print(f"{'Método':<22} {'Iterações':>10} {'R² (escala real)':>18}")
print("-" * 60)
print(f"{'Descida do Gradiente':<22} {iters_gd:>10d} {r2_gd:>18.5f}")
print(f"{'Levenberg-Marquardt':<22} {iters_lm:>10d} {r2_lm:>18.5f}")
print("=" * 60)

grid_x = np.linspace(x_raw.min(), x_raw.max(), 40)
grid_y = np.linspace(y_raw.min(), y_raw.max(), 40)
Gx, Gy = np.meshgrid(grid_x, grid_y)

Gx_zscore = (Gx - x_mean) / x_std
Gy_zscore = (Gy - y_mean) / y_std
F_bar_grid = construir_F_bar(Gx_zscore.ravel(), Gy_zscore.ravel())

Gz_gd = zscore_desnormaliza(modelo(theta_gd, F_bar_grid), z_mean, z_std).reshape(Gx.shape)
Gz_lm = zscore_desnormaliza(modelo(theta_lm, F_bar_grid), z_mean, z_std).reshape(Gx.shape)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]], subplot_titles=(f"Descida do Gradiente ({iters_gd} iterações)", f"Levenberg-Marquardt ({iters_lm} iterações)"))

fig.add_trace(go.Scatter3d(x=x_raw, y=y_raw, z=z_raw, mode="markers", marker=dict(size=2, color="black"), showlegend=False), row=1, col=1)
fig.add_trace(go.Surface(x=Gx, y=Gy, z=Gz_gd, opacity=0.7, colorscale="Blues", showscale=False), row=1, col=1)

fig.add_trace(go.Scatter3d(x=x_raw, y=y_raw, z=z_raw, mode="markers", marker=dict(size=2, color="black"), showlegend=False), row=1, col=2)
fig.add_trace(go.Surface(x=Gx, y=Gy, z=Gz_lm, opacity=0.7, colorscale="Viridis", showscale=False),row=1, col=2)

fig.update_layout(title="Comparação de Superfícies Ajustadas", height=600, width=1100)

fig.show()


Método                  Iterações   R² (escala real)
------------------------------------------------------------
Descida do Gradiente           60            0.98576
Levenberg-Marquardt             3            0.98576
